# L4 · Notebook 04 — Sweep order 对比（同步 / In-place / 优先扫描）

**对应 PPT**：`L4-DP.tex` §6 异步 DP（line 1048）+ In-place 更新与优先扫描（line 1074）

## 教学目标

DP 的"sweep 顺序"决定每轮更新哪些状态、用什么 V 值参考：

1. **同步**（Jacobi）：所有状态用上一轮的 $V_k$ 更新，得到 $V_{k+1}$。两份存储
2. **In-place** / **Gauss-Seidel**：按某个顺序遍历，新值立即写回，下一状态可能用到
3. **优先扫描**（prioritized sweeping）：按 Bellman 残差排序，先更新高残差状态

**实测点**：三种 sweep 在同一 $3\times 3$ 网格上的收敛速度。

## 结论预告

- In-place 通常 1.5–2× 快于同步（无理论保证，但实务总优）
- 优先扫描进一步加速 2–3×，但需维护残差堆（O(log n)）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'L1-mdp-foundations')))
from grid_world import ACTIONS, GridConfig, GridWorld

env = GridWorld(GridConfig(gamma=0.9))
states = list(env.all_states())
nS = len(states)
idx = {s: i for i, s in enumerate(states)}

## 1. 三种 VI sweep 实现

In [ ]:
def bellman_backup(V, s):
    """VI 一次最优 backup：max_a [r + γ V(s')]。"""
    best = -np.inf
    for a in ACTIONS:
        s_next, r, _ = env.step(s, a)
        q = r + env.cfg.gamma * V[idx[s_next]]
        if q > best:
            best = q
    return best

def vi_sync(tol=1e-8, max_iter=500):
    """同步：所有状态用上一轮 V。"""
    V = np.zeros(nS)
    history = [V.copy()]
    for k in range(max_iter):
        V_new = np.array([bellman_backup(V, s) for s in states])
        history.append(V_new.copy())
        if np.max(np.abs(V_new - V)) < tol:
            return V_new, history
        V = V_new
    return V, history

def vi_inplace(order='row', tol=1e-8, max_iter=500):
    """In-place：按 order 遍历，新值立即覆盖。
    order='row' = 行优先；order='reverse' = 反向；order='random' = 每轮随机。"""
    V = np.zeros(nS)
    history = [V.copy()]
    rng = np.random.default_rng(0)
    perm_base = list(range(nS))
    for k in range(max_iter):
        if order == 'random':
            perm = rng.permutation(perm_base)
        elif order == 'reverse':
            perm = perm_base[::-1]
        else:
            perm = perm_base
        max_delta = 0.0
        for i in perm:
            new = bellman_backup(V, states[i])
            max_delta = max(max_delta, abs(new - V[i]))
            V[i] = new  # 立即写回
        history.append(V.copy())
        if max_delta < tol:
            return V.copy(), history
    return V.copy(), history

def vi_prioritized(tol=1e-8, max_updates=5000):
    """优先扫描：每次选 |Bellman 残差| 最大的状态更新；记录每 nS 次更新一次。"""
    V = np.zeros(nS)
    history = [V.copy()]
    n_updates = 0
    for outer in range(max_updates // nS):
        # 计算所有状态的残差
        residuals = np.array([abs(bellman_backup(V, s) - V[i]) for i, s in enumerate(states)])
        if residuals.max() < tol:
            break
        # 一轮内做 nS 次更新，每次选当前残差最大的（更新后重算）
        for _ in range(nS):
            residuals = np.array([abs(bellman_backup(V, s) - V[i]) for i, s in enumerate(states)])
            j = int(np.argmax(residuals))
            V[j] = bellman_backup(V, states[j])
            n_updates += 1
        history.append(V.copy())
    return V.copy(), history

V1, hist_sync = vi_sync()
V2, hist_inplace_row = vi_inplace(order='row')
V3, hist_inplace_rev = vi_inplace(order='reverse')
V4, hist_prior = vi_prioritized()

print(f'同步：    {len(hist_sync)-1} 轮收敛')
print(f'In-place (行优先)： {len(hist_inplace_row)-1} 轮')
print(f'In-place (反向):    {len(hist_inplace_rev)-1} 轮')
print(f'优先扫描：           ~{len(hist_prior)-1} 等价轮次')

# 一致性核查
for name, V in [('sync', V1), ('inplace-row', V2), ('inplace-rev', V3), ('prior', V4)]:
    print(f'  {name:>15s}: ||V - V_sync||_inf = {np.max(np.abs(V - V1)):.2e}')

## 2. 收敛曲线对比

In [ ]:
def residuals_from_history(history):
    return [np.max(np.abs(history[k+1] - history[k])) for k in range(len(history)-1)]

fig, ax = plt.subplots(figsize=(8, 4))
for hist, label, marker in [
    (hist_sync,         '同步 (Jacobi)',          'o'),
    (hist_inplace_row,  'In-place 行优先 (GS)',   's'),
    (hist_inplace_rev,  'In-place 反向',          '^'),
    (hist_prior,        '优先扫描',                'D'),
]:
    res = residuals_from_history(hist)
    ax.semilogy(range(1, len(res)+1), res, '-' + marker, markersize=4, label=label)

ax.set_xlabel('轮次 (等价于 |S| 次单状态 backup)')
ax.set_ylabel(r'$\|V_{k+1} - V_k\|_\infty$')
ax.set_title('Sweep order 对 VI 收敛速度的影响 (3×3 网格)')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('figures/sweep_order_compare.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. 课堂诊断小结

| Sweep 类型 | 收敛轮次（3×3 例）| 实现复杂度 | 内存 |
|---|---|---|---|
| 同步 (Jacobi) | 基准 | 简单 | 2× $\|S\|$ |
| In-place (Gauss-Seidel) | 通常更少 | 简单 | $\|S\|$ |
| 优先扫描 | 进一步少 | 中（需堆）| $\|S\|$ + 堆 |

**关键观察**：

- **In-place 优势来源**：每个状态用到的 $V(s')$ 已经是"更新过"的——信息传播更快
- **行优先 vs 反向**：哪个更快取决于奖励结构。若奖励集中在 target，反向遍历"从 target 往回传"可能更快
- **优先扫描**：把计算资源集中在"价值估计变化大"的状态——典型 RTDP 思想
- 所有方法**最终都收敛到同一 $V^*$**（一致性核查通过）

## 工程意义

- 大规模 MDP 上（如 $|\mathcal S| = 10^6$），同步 VI 一轮内存 16MB，sweep 顺序选择直接决定能否完工
- L7 Q-learning **天然 in-place**（采样一条 transition 立即更新一个 $Q(s,a)$）
- L8 DQN 用经验回放 **打乱了 sweep 顺序**——本质是"random in-place"的近似

## 思考题

1. 用更大的网格（如 $10\times 10$）重跑，sweep 顺序的相对差距是变大还是变小？
2. 优先扫描的实际"加速比"在何种 MDP 结构下最高？（提示：稀疏奖励 + 长链路径）
3. In-place 反向遍历什么时候反而更慢？（提示：奖励集中在起点附近时）